# Inspect ADE20k Segmentation Dataset

# Dataset Files

In [ ]:
import os

from torch._C.cpp import nn

root_dir = "datasets/ADEChallengeData2016/ADEChallengeData2016"
images_dir = os.path.join(root_dir, "images")
annotations_dir = os.path.join(root_dir, "annotations")

images_train_dir = os.path.join(images_dir, "training")
annotations_train_dir = os.path.join(annotations_dir, "training")
images_val_dir = os.path.join(images_dir, "validation")
annotations_val_dir = os.path.join(annotations_dir, "validation")

training_images = os.listdir(images_train_dir)
validation_images = os.listdir(images_val_dir)

print(len(training_images), len(validation_images))

# Ade20k Dataset

In [ ]:
import os
from pathlib import Path
from typing import Callable, Optional

from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import (
    Compose, Resize, ToTensor, Normalize, PILToTensor, InterpolationMode
)

class ADE20KSegDataset(Dataset):
    """
    images_dir: folder with RGB *.jpg files
    masks_dir : folder with 8-bit *.png masks (same filename stem)
    """

    def __init__(
        self,
        images_dir: str | Path,
        masks_dir: str | Path,
        img_transform : Optional[Callable] = None,
        mask_transform: Optional[Callable] = None,
    ):
        self.images_dir  = Path(images_dir)
        self.masks_dir   = Path(masks_dir)
        if img_transform is None:
            img_transform = Compose([
                Resize((256, 256), interpolation=InterpolationMode.BILINEAR),
                ToTensor(),
                Normalize(mean=[0.485, 0.456, 0.406],
                          std =[0.229, 0.224, 0.225]),
            ])
        self.img_tf  = img_transform
        if mask_transform is None:
            mask_transform = Compose([
                Resize((256, 256), interpolation=InterpolationMode.NEAREST),
                PILToTensor(),  # keeps uint8 [0‒255]
            ])
        self.mask_tf = mask_transform

        self.image_files = sorted(p for p in self.images_dir.glob("*.jpg"))
        if not self.image_files:
            raise RuntimeError(f"No .jpg files in {self.images_dir}")

        # verify every image has a mask
        self.mask_files = []
        for p in self.image_files:
            m = self.masks_dir / (p.stem + ".png")
            if not m.exists():
                raise FileNotFoundError(f"Missing mask for {p.name}")
            self.mask_files.append(m)

    def __len__(self) -> int:
        return len(self.image_files)

    def __getitem__(self, idx: int):
        img  = Image.open(self.image_files[idx]).convert("RGB")
        mask = Image.open(self.mask_files[idx])  # uint8 grayscale

        if self.img_tf:
            img = self.img_tf(img)
        if self.mask_tf:
            mask = self.mask_tf(mask)

        # squeeze the single channel and cast to long (N, H, W)   → (H, W)
        mask = mask.squeeze(0).long()     # → (H, W), values in {0,1,…,150,255}

        # remap in-place:
        #
        #   raw 1–150 → 0–149
        #   raw 0 or >150 → 255 (ignore_index)
        #
        num_classes  = 150
        ignore_index = 255

        # start with all ignored
        out = torch.full_like(mask, ignore_index)

        # select valid ADE20K IDs
        valid = (mask >= 1) & (mask <= num_classes)
        out[valid] = mask[valid] - 1


        return img, out


# Define the transformations
IMG_SIZE = (256, 256)

img_transform = Compose([
    Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406],
              std =[0.229, 0.224, 0.225]),
])

mask_transform = Compose([
    Resize(IMG_SIZE, interpolation=InterpolationMode.NEAREST),
    PILToTensor(),          # keeps uint8 [0‒255]
])

train_ds = ADE20KSegDataset(
    images_dir=images_train_dir,
    masks_dir =annotations_train_dir,
    img_transform=img_transform,
    mask_transform=mask_transform,
)
val_ds = ADE20KSegDataset(
    images_dir=images_val_dir,
    masks_dir =annotations_val_dir,
    img_transform=img_transform,
    mask_transform=mask_transform,
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)



In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
import torch

# ADE20K 150-class color palette from MIT Scene Parsing Benchmark
ADE20K_COLORMAP = [
    (120, 120, 120), (180, 120, 120), (  6, 230, 230), ( 80,  50,  50), (  4, 200,   3),
    (120, 120,  80), (140, 140, 140), (204,   5, 255), (230, 230, 230), (  4, 250,   7),
    (224,   5, 255), (235, 255,   7), (150,   5,  61), (120, 120,  70), (  8, 255,  51),
    (255,   6,  82), (143, 255, 140), (204, 255,   4), (255,  51,   7), (204,  70,   3),
    (  0, 102, 200), ( 61, 230, 250), (255,   6,  51), ( 11, 102, 255), (255,   7,  71),
    (255,   9, 224), (  9,   7, 230), (220, 220, 220), (255,   9,  92), (112,   9, 255),
    (  8, 255, 214), (  7, 255, 224), (255, 184,   6), ( 10, 255,  71), (255,  41,  10),
    (  7, 255, 255), (224, 255,   8), (102,   8, 255), (255,  61,   6), (255, 194,   7),
    (255, 122,   8), (  0, 255,  20), (255,   8,  41), (255,   5, 153), (  6,  51, 255),
    (235,  12, 255), (160, 150,  20), (  0, 163, 255), (140, 140, 140), (250,  10,  15),
    ( 20, 255,   0), ( 31, 255,   0), (255,  31,   0), (255, 224,   0), (153, 255,   0),
    (  0,  0, 255), (255,  71,   0), (  0, 235, 255), (  0, 173, 255), ( 31,   0, 255),
    (255,  39,   0), (255,  60,   0), (255,   0, 133), (255, 218,   7), (255,   0, 110),
    (255,   0, 255), (  0, 255, 160), (255, 255,   0), (255, 255, 255), (  0,  63, 255),
    (255,  71,  15), (  0, 255, 182), (  0, 255, 255), (255, 230,   0), (255,   0, 150),
    (255,  20, 255), (255,   0, 200), (255, 255,  25), (  0, 255,  30), ( 40,   0, 255),
    (  0, 255,  50), ( 80,   0, 255), (  0, 255, 100), (120,   0, 255), (160,   0, 255),
    (200,   0, 255), (255,   0, 245), (255,   0, 210), (255,   0, 170), (255,   0, 130),
    (255,   0,  70), (255,   0,  30), (255,  40,   0), (255,  80,   0), (255, 120,   0),
    (255, 160,   0), (255, 200,   0), (255, 240,   0), (255, 255,   0), (200, 255,   0),
    (160, 255,   0), (120, 255,   0), ( 80, 255,   0), ( 40, 255,   0), (  0, 255,   0),
    (  0, 255,  40), (  0, 255,  80), (  0, 255, 120), (  0, 255, 160), (  0, 255, 200),
    (  0, 255, 240), (  0, 255, 255), (  0, 200, 255), (  0, 160, 255), (  0, 120, 255),
    (  0,  80, 255), (  0,  40, 255), (  0,   0, 255), ( 40,   0, 255), ( 80,   0, 255),
    (120,   0, 255), (160,   0, 255), (200,   0, 255), (255,   0, 255), (255,   0, 200),
    (255,   0, 160), (255,   0, 120), (255,   0,  80), (255,   0,  40), (255,   0,   0),
    (255,  40,   0), (255,  80,   0), (255, 120,   0), (255, 160,   0), (255, 200,   0),
    (255, 240,   0), (255, 255,   0)
]


def show_image_and_mask(image: torch.Tensor, mask: torch.Tensor, class_colors=None, class_labels=None):
    """
    image: Tensor of shape (3, H, W), normalized
    mask : Tensor of shape (H, W), with integer class indices
    class_colors: Optional list of RGB tuples (0-255) for mapping class indices
    class_labels: Optional list of class names corresponding to indices
    """
    # Denormalize the image for visualization (ImageNet stats)
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std  = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    image_vis = image * std + mean
    image_vis = torch.clamp(image_vis, 0, 1)

    # Create color mask if class_colors provided
    if class_colors is not None:
        mask_rgb = torch.zeros(3, *mask.shape, dtype=torch.uint8)
        for idx, color in enumerate(class_colors):
            mask_rgb[:, mask == idx] = torch.tensor(color, dtype=torch.uint8).view(3, 1)
        mask_vis = mask_rgb.permute(1, 2, 0).numpy()
    else:
        mask_vis = mask.numpy()

    # Plot
    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(image_vis.permute(1, 2, 0).cpu())
    axs[0].set_title("Input Image")
    axs[0].axis("off")

    if class_colors is not None:
        axs[1].imshow(mask_vis)
    else:
        im = axs[1].imshow(mask_vis, cmap="tab20", vmin=0, vmax=150)
        fig.colorbar(im, ax=axs[1], fraction=0.046, pad=0.04)

    axs[1].set_title("Segmentation Mask")
    axs[1].axis("off")

    plt.tight_layout()
    plt.show()


# Get a sample from the dataset
image, mask = train_ds[2]

# Visualize
show_image_and_mask(image, mask, class_colors=ADE20K_COLORMAP, class_labels=None)


In [ ]:
mask

In [ ]:
classes, inverse, counts = torch.unique(mask, return_counts=True, return_inverse=True)
print(f"Classes: {classes}")
print(f"Counts: {counts}")
print(f"Inverse: {inverse}")
plt.hist(inverse.cpu().numpy().flatten(), bins=150, range=(0, 150), density=True)
plt.show()
plt.bar(classes.cpu().numpy(), counts.cpu().numpy())
plt.xlabel("Class Index")
plt.ylabel("Count")
plt.title("Class Distribution in Segmentation Mask")
plt.xticks(classes.cpu().numpy())
plt.show()

plt.imshow(mask.cpu().numpy(), cmap="tab20", vmin=0, vmax=150)
plt.show()

plt.imshow(inverse.cpu().numpy(), cmap="tab20", vmin=0, vmax=150)
plt.show()


# ADE Model

In [ ]:
from collections.abc import Sequence
import os

from torch.utils.data import DataLoader
from torchvision import transforms

from torch import compile as jit
from oxels.networks import SimpleUNet
from oxels.losses import vectorized_contrastive_loss, vectorized_loss
from torch.utils.data import Dataset
import torchvision.transforms as T

from oxels.transforms import PerspectiveTransform

from oxels.models import BaseModel

class ADE20KDataset(Dataset):
    """
    images_dir: folder with RGB *.jpg files
    masks_dir : folder with 8-bit *.png masks (same filename stem)
    """

    def __init__(
        self,
        split: str = "train",
        img_transform : Optional[Callable] = None,
        mask_transform: Optional[Callable] = None,
    ):
        root_dir = "datasets/ADEChallengeData2016/ADEChallengeData2016"
        images_dir = os.path.join(root_dir, "images")
        annotations_dir = os.path.join(root_dir, "annotations")
        if split == "train":
            images_dir = os.path.join(images_dir, "training")
            annotations_dir = os.path.join(annotations_dir, "training")
        elif split == "val":
            images_dir = os.path.join(images_dir, "validation")
            annotations_dir = os.path.join(annotations_dir, "validation")
        else:
            raise ValueError(f"Unknown split: {split}")
        self.images_dir  = Path(images_dir)
        self.masks_dir   = Path(annotations_dir)
        if img_transform is None:
            img_transform = Compose([
                Resize((256, 256), interpolation=InterpolationMode.BILINEAR),
                ToTensor(),
                Normalize(mean=[0.485, 0.456, 0.406],
                          std =[0.229, 0.224, 0.225]),
            ])
        self.img_tf  = img_transform
        if mask_transform is None:
            mask_transform = Compose([
                Resize((256, 256), interpolation=InterpolationMode.NEAREST),
                PILToTensor(),  # keeps uint8 [0‒255]
            ])
        self.mask_tf = mask_transform

        self.image_files = sorted(p for p in self.images_dir.glob("*.jpg"))
        if not self.image_files:
            raise RuntimeError(f"No .jpg files in {self.images_dir}")

        # verify every image has a mask
        self.mask_files = []
        for p in self.image_files:
            m = self.masks_dir / (p.stem + ".png")
            if not m.exists():
                raise FileNotFoundError(f"Missing mask for {p.name}")
            self.mask_files.append(m)

    def __len__(self) -> int:
        return len(self.image_files)

    def __getitem__(self, idx: int):
        img  = Image.open(self.image_files[idx]).convert("RGB")
        mask = Image.open(self.mask_files[idx])  # uint8 grayscale

        if self.img_tf:
            img = self.img_tf(img)
        if self.mask_tf:
            mask = self.mask_tf(mask)

        # squeeze the single channel and cast to long (N, H, W)   → (H, W)
        mask = mask.squeeze(0).long()     # → (H, W), values in {0,1,…,150,255}

        # remap in-place:
        #
        #   raw 1–150 → 0–149
        #   raw 0 or >150 → 255 (ignore_index)
        #
        num_classes  = 150
        ignore_index = 255

        # start with all ignored
        out = torch.full_like(mask, ignore_index)

        # select valid ADE20K IDs
        valid = (mask >= 1) & (mask <= num_classes)
        out[valid] = mask[valid] - 1
        return img, out





class ADE20KBackboneDataset(Dataset):
    default_augmentations = T.Compose([
        T.ToPILImage(),
        T.ColorJitter(brightness=0.3, hue=(-0.1, 0.1), saturation=0.3),
        T.RandomAutocontrast(p=0.1),
        T.RandomApply([T.GaussianBlur(kernel_size=7, sigma=(1.0, 3.0))], p=0.1),
        T.RandomPosterize(5, p=0.1),
        T.RandomEqualize(p=0.1),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
    ])

    def __init__(
        self,
        split: str = "train",
        w: int = 256,
        h: int = 256,
        frac_keep: float = 0.125,
        augmentations=default_augmentations,
        img_transform=None,
    ):
        super().__init__()
        self.split = split
        self.dataset = ADE20KDataset(self.split, img_transform=img_transform)
        self.perspective_transform = PerspectiveTransform(w=w, h=h, frac_keep=frac_keep)
        self.augmentations = augmentations

    def __getitem__(self, item):
        with torch.device("cpu"):
            rgb, segmentation = self.dataset[item]
        rgb = rgb.numpy()
        rgb = rgb.transpose(1, 2, 0)

        # this needs channels last
        view1, view2, permutation, flags, mask1, mask2 = self.perspective_transform.get_views_and_permutation(rgb)

        view1 = view1.transpose(2, 0, 1)
        view2 = view2.transpose(2, 0, 1)

        with torch.device("cpu"):
            view1 = torch.from_numpy(view1).float()
            view2 = torch.from_numpy(view2).float()

        view1 = self.augmentations(view1)
        view2 = self.augmentations(view2)

        return view1, view2, permutation, flags, mask1, mask2

    def __len__(self):
        return len(self.dataset)



class Ade20kModel(BaseModel):
    def __init__(
        self,
        *,
        stage_channels: Sequence[int] = (32, 64, 128, 256),
        num_res_blocks: Sequence[int] = (2, 2, 2, 4),
        num_oxels: int = 64,
        num_norm_groups: int = 8,
        learning_rate: float = 1e-3,
        weight_decay: float = 0.004,
        dropout_stages: Sequence[int],
        dropout: float = 0.1,
        attention_stages: Sequence[int],
        lr_div_factor: float = 25.0,
        lr_final_div_factor: float = 1e4,
        lr_pct_start: float = 0.05,
        train_batch_size: int,
        val_batch_size: int,
        image_size: int = 256,
        frac_keep: float = 0.25,
        contrastive_loss_weight: float = 0.5,
    ):
        num_stages = len(stage_channels)
        has_attention = [False] * num_stages
        for stage in attention_stages:
            has_attention[stage] = True

        residual_dropout = [0.0] * num_stages
        for stage in dropout_stages:
            residual_dropout[stage] = dropout

        backbone = SimpleUNet(
            height=image_size,
            width=image_size,
            in_channels=3,
            out_channels=num_oxels,
            channels_of_stage=stage_channels,
            has_attention=has_attention,
            num_res_blocks=num_res_blocks,
            norm_groups=num_norm_groups,
            residual_dropout=residual_dropout,
        )

        super().__init__(
            backbone=backbone,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            lr_div_factor=lr_div_factor,
            lr_final_div_factor=lr_final_div_factor,
            lr_pct_start=lr_pct_start,
            contrastive_loss_weight=contrastive_loss_weight,
        )

        self.save_hyperparameters()

    @jit
    def compute_loss_test(self, batch):
        view1, view2, permutation, flags, mask1, mask2 = batch

        oxels_view1 = self(view1)
        oxels_view2 = self(view2)

        loss = vectorized_loss(oxels_view1, oxels_view2, permutation, flags, mask1, mask2)

        c = 0.5#self.hparams.contrastive_loss_weight
        if c > 0.0:
            closs = vectorized_contrastive_loss(oxels_view1, oxels_view2, permutation, mask1, mask2)
            loss = (1 - c) * loss + c * closs

        return loss

    @jit
    def compute_loss_validation(self, batch):
        view1, view2, permutation, flags, mask1, mask2 = batch

        oxels_view1 = self(view1)
        oxels_view2 = self(view2)

        loss = vectorized_loss(oxels_view1, oxels_view2, permutation, flags, mask1, mask2)

        c = 0.5#self.hparams.contrastive_loss_weight
        if c > 0.0:
            closs = vectorized_contrastive_loss(oxels_view1, oxels_view2, permutation, mask1, mask2)
            loss = (1 - c) * loss + c * closs

        return loss


    def train_dataloader(self):
        img_transform = Compose([
                Resize((self.hparams.image_size, self.hparams.image_size), interpolation=InterpolationMode.BILINEAR),
                ToTensor(),
                Normalize(mean=[0.485, 0.456, 0.406],
                          std =[0.229, 0.224, 0.225]),
                transforms.RandomHorizontalFlip(),
                transforms.RandomResizedCrop(self.hparams.image_size, scale=(0.7, 1.0)),
            ])
        dataset = ADE20KBackboneDataset(
            h=self.hparams.image_size,
            w=self.hparams.image_size,
            split="train",
            frac_keep=self.hparams.frac_keep,
            img_transform = img_transform,
        )
        return DataLoader(
            dataset,
            batch_size=self.hparams.train_batch_size,
            shuffle=True,
            pin_memory=True,
            num_workers=len(os.sched_getaffinity(0)),
            drop_last=True,
        )

    def val_dataloader(self):
        img_transform = Compose([
                Resize((self.hparams.image_size, self.hparams.image_size), interpolation=InterpolationMode.BILINEAR),
                ToTensor(),
                Normalize(mean=[0.485, 0.456, 0.406],
                          std =[0.229, 0.224, 0.225]),
                transforms.RandomHorizontalFlip(),
                transforms.RandomResizedCrop(self.hparams.image_size, scale=(0.7, 1.0)),
            ])
        val_id = ADE20KBackboneDataset(
            h=self.hparams.image_size,
            w=self.hparams.image_size,
            split="val",
            frac_keep=0.25,
            img_transform=img_transform,
        )

        return DataLoader(
            val_id,
            batch_size=self.hparams.val_batch_size,
            shuffle=False,
            pin_memory=True,
            num_workers=len(os.sched_getaffinity(0)),
            drop_last=False,
        )

# load backbone

In [ ]:
import os
from oxels.models import ImageNetModel


#backbone_run_name = "last.ckpt"
#backbone_run_name = "scarlet-sky-138"
#backbone_run_name = "cluster-no-contrastive-loss.ckpt"
backbone_run_name = "wobbly-morning-60"
#ckpt_dir = os.path.join("../checkpoints")
ckpt_dir = os.path.join("Ade20k Backbone Trainings", "lightning_logs", str(backbone_run_name))
with open(os.path.join(ckpt_dir, "wandb_run_id.txt"), "r") as f:
    backbone_run_id = f.read().strip()
backbone_ckpt_path = os.path.join("Ade20k Backbone Trainings/lightning_logs", backbone_run_name, "last.ckpt")
#backbone_ckpt_path = os.path.join("../checkpoints",backbone_run_name)
backbone = Ade20kModel.load_from_checkpoint(backbone_ckpt_path)
print(f"Loaded model from {os.path.join("imagenet_backbones", backbone_run_name)} run {backbone_run_id}")
num_oxels = backbone.hparams.num_oxels

# Oxel-loading Dataset

In [ ]:
import os
from pathlib import Path
from typing import Callable, Optional

from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import (
    Compose, Resize, ToTensor, Normalize, PILToTensor, InterpolationMode
)
from multiprocessing import get_context

class ADE20KOxelSegDataset(Dataset):
    """
    images_dir: folder with RGB *.jpg files
    masks_dir : folder with 8-bit *.png masks (same filename stem)
    """

    def __init__(
        self,
        backbone,
        device,
        images_dir: str | Path,
        masks_dir: str | Path,
        img_transform : Optional[Callable] = None,
        mask_transform: Optional[Callable] = None,
        num_pixels:    int = 100,
        num_classes:    int = 150,
        ignore_index:   int = 255,

    ):
        self.segmentation_dataset = ADE20KSegDataset(
            images_dir=images_dir,
            masks_dir=masks_dir,
            img_transform=img_transform,
            mask_transform=mask_transform,
        )
        self.device = torch.device(device)
        self.backbone = backbone.to(self.device)
        self.backbone.freeze()
        self.num_pixels  = num_pixels
        self.num_classes = num_classes
        self.ignore_idx  = ignore_index


    def __len__(self) -> int:
        return len(self.segmentation_dataset)

    def __getitem__(self, idx: int):
        img, mask = self.segmentation_dataset[idx]

        # 1) get per-pixel features
        x = img.unsqueeze(0).to(self.device)    # (1,3,H,W)
        feat = self.backbone(x).squeeze(0)   # (C, h, w)
        feat = feat.cpu()
        C, h, w = feat.shape

        # 2) flatten both
        flat_feat = feat.view(C, -1)            # (C, H*W)
        flat_mask = mask.view(-1)               # (H*W,)

        # 3) restrict to non-ignored pixels
        valid = flat_mask != self.ignore_idx    # (H*W,)
        valid_inds  = torch.nonzero(valid, as_tuple=True)[0]   # (M,)
        valid_labels = flat_mask[valid]                        # (M,)

        # 4) if no valid pixels, just sample randomly from entire image
        if valid_inds.numel() == 0:
            all_inds   = torch.randperm(h*w)[: self.num_pixels]
            feat_sel   = flat_feat[:, all_inds].unsqueeze(-1)  # (C, N, 1)
            label_sel  = flat_mask[all_inds].unsqueeze(-1)                   # (N,1)
            return feat_sel, label_sel

        # 4) find which classes appear & get inverse map
        classes, inv = torch.unique(valid_labels,
                                    return_inverse=True)       # classes:(K,), inv:(M,)
        K = classes.numel()
        per_cls = self.num_pixels // K

        # 5) sample per class via inv
        picks = []
        for cls_idx in range(K):
            cls_positions = valid_inds[inv == cls_idx]         # all positions of this class
            n_pos = cls_positions.numel()
            if n_pos >= per_cls:
                choices = cls_positions[torch.randperm(n_pos)[:per_cls]]
            else:
                # sample with replacement if too few
                choices = cls_positions[
                    torch.randint(0, n_pos, (per_cls,), dtype=torch.long)
                ]
            picks.append(choices)
        picks = torch.cat(picks)                                # (~K*per_cls,)

        # 6) if we still need more to reach num_pixels, pad from valid_inds
        if picks.numel() < self.num_pixels:
            extra_needed = self.num_pixels - picks.numel()
            extra = valid_inds[torch.randperm(valid_inds.numel())[:extra_needed]]
            picks = torch.cat([picks, extra])

        # 7) truncate and gather features + labels
        picks = picks[: self.num_pixels]                       # exactly N
        feat_samples = flat_feat[:, picks].unsqueeze(-1)       # (C, N, 1)
        labels       = flat_mask[picks].unsqueeze(-1)                        # (N, 1)

        return feat_samples, labels



In [ ]:
# Define the transformations
IMG_SIZE = (256, 256)

img_transform = Compose([
    Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406],
              std =[0.229, 0.224, 0.225]),
])

mask_transform = Compose([
    Resize(IMG_SIZE, interpolation=InterpolationMode.NEAREST),
    PILToTensor(),          # keeps uint8 [0‒255]
])

train_ds = ADE20KOxelSegDataset(
    backbone=backbone,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_pixels=100,
    num_classes=150,
    ignore_index=255,
    images_dir=images_train_dir,
    masks_dir =annotations_train_dir,
    img_transform=img_transform,
    mask_transform=mask_transform,
)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0, pin_memory=False)

feats, labels = next(iter(train_loader))
print(f"Features shape: {feats.shape}")  # (C, N, 1)
print(f"Labels shape: {labels.shape}")    # (N,)


# Test Segmentation Proxy Loss Callbacck

In [ ]:
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = 256

# Load a sample batch from the ADE20K dataset
img_transform = Compose([
    Resize((image_size, image_size), interpolation=InterpolationMode.BILINEAR),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406],
              std =[0.229, 0.224, 0.225]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(image_size, scale=(0.7, 1.0)),
])
val_id = ADE20KDataset(
    split="val",
)
val_id_dataloader = DataLoader(
    val_id,
    batch_size=8,  # Use a smaller batch size for testing
    shuffle=False,
    pin_memory=True,
    num_workers=2,
    drop_last=False,
)

all_boxels, all_labels = [], []
image_counter, image_max = 0, 1000
for images, labels in tqdm(val_id_dataloader):
    image_counter += images.shape[0]

    images = images.to(device)
    labels = labels.to(device).long().squeeze(1)
    with torch.no_grad():
        oxels = backbone.backbone(images) # (batch_size, num_oxels, height, width)
    # to cpu and numpy
    oxels_n = oxels.cpu().numpy()
    num_oxels = oxels_n.shape[1]
    boxels = sum([2**i*np.array(oxels_n[:,i] > 0, dtype=np.uint32) for i in range(num_oxels)])
    all_boxels.append(boxels)
    all_labels.append(labels.cpu().numpy())
    if image_counter > image_max:
        break
all_boxels = np.concatenate(all_boxels, axis=0)[:image_max]
all_labels = np.concatenate(all_labels, axis=0)[:image_max]
print()

In [ ]:
uxels, counts = np.unique(all_boxels, return_counts=True)
suxels = uxels[np.argsort(counts)]
sounts = np.sort(counts)
plt.figure(figsize=(10, 5))
plt.plot(sounts, label="Counts of unique boxels")
plt.xlabel("Unique Boxels (sorted)")
plt.ylabel("Count")
plt.yscale("log")
plt.show()

In [ ]:
ADE20K_CLASS_LABELS = {
    0: 'wall', 1: 'building', 2: 'sky', 3: 'floor', 4: 'tree', 5: 'ceiling', 6: 'road', 7: 'bed',
    8: 'windowpane', 9: 'grass', 10: 'cabinet', 11: 'sidewalk', 12: 'person', 13: 'earth', 14: 'door',
    15: 'table', 16: 'mountain', 17: 'plant', 18: 'curtain', 19: 'chair', 20: 'car', 21: 'water',
    22: 'painting', 23: 'sofa', 24: 'shelf', 25: 'house', 26: 'sea', 27: 'mirror', 28: 'rug',
    29: 'field', 30: 'armchair', 31: 'seat', 32: 'fence', 33: 'desk', 34: 'rock', 35: 'wardrobe',
    36: 'lamp', 37: 'bathtub', 38: 'railing', 39: 'cushion', 40: 'base', 41: 'box', 42: 'column',
    43: 'signboard', 44: 'chest of drawers', 45: 'counter', 46: 'sand', 47: 'sink', 48: 'skyscraper',
    49: 'fireplace', 50: 'refrigerator', 51: 'grandstand', 52: 'path', 53: 'stairs', 54: 'runway',
    55: 'case', 56: 'pool table', 57: 'pillow', 58: 'screen door', 59: 'stairway', 60: 'river',
    61: 'bridge', 62: 'bookcase', 63: 'blind', 64: 'coffee table', 65: 'toilet', 66: 'flower',
    67: 'book', 68: 'hill', 69: 'bench', 70: 'countertop', 71: 'stove', 72: 'palm', 73: 'kitchen island',
    74: 'computer', 75: 'swivel chair', 76: 'boat', 77: 'bar', 78: 'arcade machine', 79: 'hovel',
    80: 'bus', 81: 'towel', 82: 'light', 83: 'truck', 84: 'tower', 85: 'chandelier', 86: 'awning',
    87: 'streetlight', 88: 'booth', 89: 'television receiver', 90: 'airplane', 91: 'dirt track',
    92: 'apparel', 93: 'pole', 94: 'land', 95: 'bannister', 96: 'escalator', 97: 'ottoman',
    98: 'bottle', 99: 'buffet', 100: 'poster', 101: 'stage', 102: 'van', 103: 'ship', 104: 'fountain',
    105: 'conveyer belt', 106: 'canopy', 107: 'washer', 108: 'plaything', 109: 'swimming pool',
    110: 'stool', 111: 'barrel', 112: 'basket', 113: 'waterfall', 114: 'tent', 115: 'bag',
    116: 'minibike', 117: 'cradle', 118: 'oven', 119: 'ball', 120: 'food', 121: 'step', 122: 'tank',
    123: 'trade name', 124: 'microwave', 125: 'pot', 126: 'animal', 127: 'bicycle', 128: 'lake',
    129: 'dishwasher', 130: 'screen', 131: 'blanket', 132: 'sculpture', 133: 'hood', 134: 'sconce',
    135: 'vase', 136: 'traffic light', 137: 'tray', 138: 'ashcan', 139: 'fan', 140: 'pier',
    141: 'crt screen', 142: 'plate', 143: 'monitor', 144: 'bulletin board', 145: 'shower',
    146: 'radiator', 147: 'glass', 148: 'clock', 149: 'flag'
}

uannots, counts = np.unique(all_labels, return_counts=True)
for ua in uannots:
    print(ua, np.sum((all_labels == ua) & (all_boxels == suxels[-1])), ADE20K_CLASS_LABELS[ua] if ua in ADE20K_CLASS_LABELS else "NA")

In [ ]:

tp_ratios = {}
for SAMPLE_LABEL, _ in tqdm(ADE20K_CLASS_LABELS.items()):
    sample_boxels = all_boxels[np.where(all_labels == SAMPLE_LABEL)]
    unique_sample_boxels, sample_counts = np.unique(sample_boxels, return_counts=True)
    unique_sample_boxels = unique_sample_boxels[np.argsort(sample_counts)]
    sample_counts = np.sort(sample_counts)
    true_positives = 0
    all_positives = 0
    for i in range(-1,-11,-1): #go over 10 most frequent oxels associated with the sample label
        true_positives += sample_counts[i]
        b = unique_sample_boxels[i]
        all_positives += np.sum(all_boxels == b)
    tp_ratios[SAMPLE_LABEL] = true_positives / all_positives if all_positives > 0 else 0
    #print(f"{ADE20K_CLASS_LABELS[SAMPLE_LABEL]}: {true_positives/all_positives:.3f}")
plt.figure(figsize=(20, 5))
plt.bar(range(len(tp_ratios)), list(tp_ratios.values()), align='center')
plt.xticks(range(len(tp_ratios)), list(ADE20K_CLASS_LABELS.values()), rotation=90)
plt.xlabel("ADE20K Class Labels")
plt.ylabel("True Positive Ratio")
plt.title("True Positive Ratios for ADE20K Class Labels")
plt.tight_layout()
plt.show()


## Pixel Classifiers

In [ ]:
import torch
import torch.nn as nn
from torchmetrics.classification import Accuracy, JaccardIndex, Recall
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from oxels.models.metrics_mixin import MetricsMixin
import torch.nn.functional as F
import lightning as L


class Simple1x1Classifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 1, head_groups: int = 1):
        super().__init__()
        self.num_classes = num_classes
        self.linear = nn.Linear(in_channels, num_classes)

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = self.linear(x)
        x = x.permute(0, 3, 1, 2)  # [B, C, H, W]
        return x

class MLPStefanClassifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 1):
        super().__init__()
        self.num_classes = num_classes
        '''
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 150),
            nn.ReLU(),
            nn.Linear(150, 100),
            nn.ReLU(),
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, num_classes),
        )
        '''
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, 512),
            nn.GELU(),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, 256),
            nn.GELU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 3, 1)  # [B, H, W, C]
        x = self.mlp(x)
        x = x.permute(0, 3, 1, 2)  # [B, C, H, W]
        return x

class Conv1x1Classifier(nn.Module):
    def __init__(self, in_channels: int, num_classes: int = 1, hidden_channels: int = 64, depth: int = 4, dropout: float = 0.1):
        super().__init__()
        self.num_classes = num_classes
        layers = []
        layers += [
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1, bias=True),
            nn.GroupNorm(1, hidden_channels),
            nn.GELU(),
            nn.Dropout(dropout),
        ]

        for _ in range(depth - 1):
            layers += [
                nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1, bias=True),
                nn.GroupNorm(1, hidden_channels),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
        layers += [
            nn.Conv2d(hidden_channels, num_classes, kernel_size=1, bias=True),
        ]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv(x)
        return x


class DGLinearClassifier(MetricsMixin, L.LightningModule):
    def __init__(
        self,
        backbone: nn.Module,
        head: nn.Module,
        dataset_name: str,
        train_batch_size: int,
        val_batch_size: int,
        num_oxels: int = 64,
        image_size: int = 256,
        weight_decay: float = 0.004,
        learning_rate: float = 1e-3,
        lr_pct_start: float = 0.05,
        lr_div_factor: float = 25.0,
        lr_final_div_factor: float = 1e4,
        num_trai_pixels: int = 100,
    ):
        super().__init__()
        self.save_hyperparameters(
            ignore=["backbone", "head", "dataset_name"],
        )
        self.backbone = backbone
        self.backbone.freeze()  # freeze the backbone
        self.head = head
        self.dataset_name = dataset_name
        num_classes = head.num_classes
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=255)
        self.num_train_pixels = num_trai_pixels

        self.train_acc = Accuracy(task="multiclass", num_classes=num_classes, average='micro', ignore_index=255)
        self.val_acc = Accuracy(task="multiclass", num_classes=num_classes, average='micro', ignore_index=255)
        self.test_acc = Accuracy(task="multiclass", num_classes=num_classes, average='micro', ignore_index=255)

        self.train_miou_metric = JaccardIndex(task="multiclass", num_classes=num_classes, average="macro", ignore_index=255)
        self.val_miou_metric = JaccardIndex(task="multiclass", num_classes=num_classes, average="macro", ignore_index=255)
        self.test_miou_metric = JaccardIndex(task="multiclass", num_classes=num_classes, average="macro", ignore_index=255)

        self.train_macc = Recall(task='multiclass', num_classes=num_classes, average='macro', ignore_index=255)
        self.val_macc = Recall(task='multiclass', num_classes=num_classes, average='macro', ignore_index=255)
        self.test_macc = Recall(task='multiclass', num_classes=num_classes, average='macro', ignore_index=255)

    def forward(self, batch):
        # B, C, M, 1 or B, C, H, W
        if self.trainer.training:
            prediction = self.forward_with_oxel(batch)
        else:
            prediction = self.forward_with_image(batch)
        return prediction

    def forward_with_image(self, image_batch):
        """
        Forward pass with the backbone and head.
        This is useful for computing features from the backbone.
        """
        oxel_batch = self.backbone(image_batch)  # BxCxHxW
        prediction = self.forward_with_oxel(oxel_batch)
        return prediction

    def forward_with_oxel(self, oxel_batch):
        """
        Forward pass with the backbone and head.
        This is useful for computing features from the backbone.
        """
        prediction = self.head(oxel_batch)
        return prediction

    def compute_loss(self, batch):
        images, labels = batch # Bx3xHxW,  Bx1xHxW
        logits = self(images) # bx21xhxw
        #labels = labels.to(dtype=logits.dtype).squeeze(1) # BxHxW
        #labels = labels.squeeze(1).long()  # BxHxW, convert to long for CrossEntropyLoss
        #labels = labels.to(dtype=logits.dtype)
        loss = self.loss_fn(logits, labels)
        return loss

    def compute_metrics(self, batch):
        images, labels = batch
        logits = self(images)
        B, C, H, W = logits.shape
        assert C == self.head.num_classes, f"Got {C} output channels, expected {self.head.num_classes}"
        assert labels.dtype == torch.long, f"Labels dtype {labels.dtype} != long"
        assert labels.shape == (B, H, W), f"Labels shape {labels.shape} does not match logits shape {logits.shape}"

        mx = int(labels.max())
        mn = int(labels.min())
        assert mn >= 0, f"Min label {mn} < 0"
        assert mx < C or mx == 255, f"Max label {mx} out of [0,{C-1}] or 255"

        # Resize labels to match logits spatial dimensions
        #labels_resized = F.interpolate(labels.float(), size=(H, W), mode="nearest").squeeze(1).long()
        #labels = labels.squeeze(1).long()
        loss = self.loss_fn(logits, labels)

        preds = torch.argmax(logits, dim=1) # BxHxW

        # choose the right Accuracy object based on stage
        if self.trainer.training:
            acc = self.train_acc(preds, labels)
            miou = self.train_miou_metric(preds, labels)
            macc = self.train_macc(preds, labels)
        elif self.trainer.validating:
            acc = self.val_acc(preds, labels)
            miou = self.val_miou_metric(preds, labels)
            macc = self.val_macc(preds, labels)
        else:  # testing
            #raise RuntimeError("Testing is not supported in this module. Use validation metrics instead.")
            acc = self.test_acc(preds, labels)
            miou = self.test_miou_metric(preds, labels)
            macc = self.test_macc(preds, labels)

        return {
            "loss": loss,
            "accuracy": acc,
            "miou": miou,
            "macc": macc,
        }

    def configure_optimizers(self):
        lr = self.hparams.learning_rate
        wd = self.hparams.weight_decay

        # only use parameters that requires grad
        params = [p for p in self.parameters() if p.requires_grad]
        optimizer = AdamW(params, lr=lr, weight_decay=wd, betas=(0.9, 0.99))
        scheduler = OneCycleLR(
            optimizer,
            max_lr=lr,
            total_steps=self.trainer.estimated_stepping_batches,
            div_factor=self.hparams.lr_div_factor,
            final_div_factor=self.hparams.lr_final_div_factor,
            pct_start=self.hparams.lr_pct_start,
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        metrics = self.compute_metrics(batch)
        prefix = "validation/" if dataloader_idx == 0 else "validation/ood/"
        for key, value in metrics.items():
            key = f"{prefix}{key}"
            self.log(key,
                     value,
                     on_step=False,
                     on_epoch=True,
                     sync_dist=True,
                     prog_bar=(dataloader_idx == 0),  # maybe only show ID in prog bar
                     logger=True)

        return metrics["loss"]

    def train_dataloader(self):
        root_dir = "datasets/ADEChallengeData2016/ADEChallengeData2016"
        images_dir = os.path.join(root_dir, "images")
        annotations_dir = os.path.join(root_dir, "annotations")

        images_train_dir = os.path.join(images_dir, "training")
        annotations_train_dir = os.path.join(annotations_dir, "training")

        train_dataset = ADE20KOxelSegDataset(
            backbone=self.backbone,
            device="cuda" if torch.cuda.is_available() else "cpu",
            num_pixels=self.num_train_pixels,
            num_classes=150,
            ignore_index=255,
            images_dir=images_train_dir,
            masks_dir =annotations_train_dir,
        )

        return DataLoader(
            train_dataset,
            batch_size=self.hparams.train_batch_size,
            shuffle=True,
            pin_memory=False,
            num_workers=0,#len(os.sched_getaffinity(0)),
            drop_last=True,
        )

    def val_dataloader(self):
        root_dir = "datasets/ADEChallengeData2016/ADEChallengeData2016"
        images_dir = os.path.join(root_dir, "images")
        annotations_dir = os.path.join(root_dir, "annotations")

        images_val_dir = os.path.join(images_dir, "validation")
        annotations_val_dir = os.path.join(annotations_dir, "validation")
        val_dataset = ADE20KSegDataset(
            images_dir=images_val_dir,
            masks_dir=annotations_val_dir,
        )
        return DataLoader(
            val_dataset,
            batch_size=self.hparams.val_batch_size,
            shuffle=False,
            pin_memory=True,
            num_workers=len(os.sched_getaffinity(0)),
            drop_last=False
        )

# Training

## Setup

In [ ]:
import torch
from lightning.pytorch import callbacks
from lightning.pytorch.loggers import WandbLogger
from torchmetrics import Accuracy
import lightning as L
from lightning.pytorch.loggers import WandbLogger

import wandb
from oxels.callbacks import ShowOxels


torch.cuda.empty_cache()
torch.set_float32_matmul_precision("medium")

print("Device Count:", torch.cuda.device_count())
project = "ADE20k-testing"
num_classes = 150
total_steps = 100_000
image_size = 256
num_nodes = 1
num_devices = 1
train_batch_size = 32
val_batch_size = 32
learning_rate = 1e-3
lr_pct_start = 0.1
lr_div_factor = 100
weight_decay = 1e-6

num_train_pixels = 512

head = Conv1x1Classifier(
    in_channels=num_oxels,
    num_classes=num_classes,
    hidden_channels=196,
    depth=4,
    dropout=0.1
)

model_config = dict(
    num_oxels=num_oxels,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    lr_pct_start=lr_pct_start,
    lr_div_factor=lr_div_factor,
    lr_final_div_factor=1e3,
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    image_size=image_size,
    num_trai_pixels=num_train_pixels,
)

trainer_config = dict(
    gradient_clip_val=3.0,
    gradient_clip_algorithm="value",
    max_steps=total_steps,
    accelerator="gpu",
    strategy="auto",
    devices=num_devices,
    precision="16-mixed",
    num_nodes=num_nodes,
)

model = DGLinearClassifier(
    backbone=backbone,
    head=head,
    dataset_name="ADE20k",
    train_batch_size=train_batch_size,
    val_batch_size=val_batch_size,
    num_oxels=num_oxels,
    image_size=image_size,
    weight_decay=weight_decay,
    learning_rate=learning_rate,
    lr_pct_start=lr_pct_start,
    lr_div_factor=lr_div_factor,
    lr_final_div_factor=1e4,
    num_trai_pixels=num_train_pixels
)

with torch.device("cpu"):
    #train_images = [model.train_dataloader().dataset[i][0] for i in range(4)]
    #train_images = torch.stack(train_images)
    validation_images = [model.val_dataloader().dataset[i][0] for i in range(4)]
    validation_images = torch.stack(validation_images)
    #train_labels = [model.train_dataloader().dataset[i][1] for i in range(4)]
    #train_labels = torch.stack(train_labels)
    validation_labels = [model.val_dataloader().dataset[i][1] for i in range(4)]
    validation_labels = torch.stack(validation_labels)

num_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total parameters: ", num_parameters)
config = model_config | trainer_config

## Init WandB and Start Training

In [ ]:
from oxels.callbacks import ShowSegmentationWandb
import torch._dynamo
torch._dynamo.config.suppress_errors = True

run = wandb.init(
    entity="kl_divergence-rensselaer-polytechnic-institute",
    project=project,
    config=config,
    dir="wandb_results"
)

logger = WandbLogger(
    experiment=run,
)

wandb.summary["num_parameters"] = num_parameters
wandb.summary["backbone_run_id"] = backbone_run_id#"synthetic_VOCS_1.0"
wandb.summary["backbone_run_name"] = backbone_run_name
wandb.summary["model_type"] = "PxSampleConv1x1Classifier"
run_name = wandb.run.name
run_id = wandb.run.id
print(f"name: {run_name} \t run id:{run_id}")

ckpt_dir = os.path.join("lightning_logs", str(run_name))
# make sure the directory exists
os.makedirs(ckpt_dir, exist_ok=True)
with open(os.path.join(ckpt_dir, "wandb_run_id.txt"), "w") as f:
    f.write(run_id)
trainer = L.Trainer(
    **trainer_config,
    callbacks=[
        callbacks.LearningRateMonitor(logging_interval="step"),
        callbacks.ModelCheckpoint(
            dirpath=ckpt_dir,
            monitor="validation/loss",
            mode="min",
            save_top_k=1,
            filename="best_model",
            save_last=True,
        ),
        #ShowSegmentationWandb(images=train_images, labels=train_labels, every_n_epochs=2, caption="Train Segmentations"),
        ShowSegmentationWandb(images=validation_images, labels=validation_labels, every_n_epochs=2, caption="Validation Segmentations"),
    ],
    logger=logger,
)

try:
    trainer.fit(model)
    result = trainer.callback_metrics["validation/loss"]
    wandb.summary["result"] = result

finally:
    # clean up
    wandb.finish()

In [ ]:
torch.cuda.empty_cache()

reduce number of batchsizes

trainin with contrastive loss backbone

stop using relu for binary input

crank up mlp once bathsize went down

subsample the pixels to increase batch size and thus variance of classes per batch

eventually need to precompute oxels to have batchsize of 1000 possible (eventualy need to binarize oxels)